In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

In [4]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

test_ids = test_df['PassengerId'].copy()

print(f'Train shape : {train_df.shape}')
print(f'Test shape  : {test_df.shape}')
train_df.head()

Train shape : (891, 12)
Test shape  : (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# EDA
train_df.info()
# Find Missing
print(train_df.isnull().sum())

train_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
# Feature Engineering
def preprocess(df, is_train=True):
    """
    Full preprocessing pipeline:
    - Drop irrelevant columns
    - Extract title from Name
    - Fill missing values
    - Encode categoricals
    - Feature engineering
    """
    data = df.copy()

    # ── 4.1 Extract Title from Name ──────────────────────────────
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    rare_titles = ['Dr','Rev','Col','Major','Mlle','Countess','Capt',
                   'Ms','Lady','Jonkheer','Don','Dona','Mme','Sir']
    data['Title'] = data['Title'].replace(rare_titles, 'Rare')
    data['Title'] = data['Title'].replace({'Miss': 'Miss', 'Mrs': 'Mrs',
                                           'Mr': 'Mr', 'Master': 'Master'})

    # ── 4.2 Family Features ──────────────────────────────────────
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone']    = (data['FamilySize'] == 1).astype(int)

    # ── 4.3 Fill Missing Age (median by Title + Pclass) ──────────
    age_median = data.groupby(['Title', 'Pclass'])['Age'].transform('median')
    data['Age'] = data['Age'].fillna(age_median)
    data['Age'] = data['Age'].fillna(data['Age'].median())  # fallback

    # ── 4.4 Fill Missing Fare ────────────────────────────────────
    data['Fare'] = data['Fare'].fillna(data.groupby('Pclass')['Fare'].transform('median'))

    # ── 4.5 Fill Missing Embarked ────────────────────────────────
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

    # ── 4.6 Age & Fare Binning ───────────────────────────────────
    data['AgeBand']  = pd.cut(data['Age'],  bins=[0,12,18,35,60,100],
                               labels=[0,1,2,3,4]).astype(int)
    data['FareBand'] = pd.qcut(data['Fare'], q=4,
                                labels=[0,1,2,3], duplicates='drop').astype(int)

    # ── 4.7 Encode Categoricals ──────────────────────────────────
    data['Sex']      = LabelEncoder().fit_transform(data['Sex'])   # female=0, male=1
    data['Embarked'] = LabelEncoder().fit_transform(data['Embarked'])
    data['Title']    = LabelEncoder().fit_transform(data['Title'])

    # ── 4.8 Cabin indicator ──────────────────────────────────────
    data['HasCabin'] = data['Cabin'].notna().astype(int)

    # ── 4.9 Drop columns ─────────────────────────────────────────
    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch']
    data.drop(columns=[c for c in drop_cols if c in data.columns], inplace=True)

    return data


# Apply preprocessing
train_processed = preprocess(train_df, is_train=True)
test_processed  = preprocess(test_df,  is_train=False)

print(f'Train features: {train_processed.shape}')
print(f'Test features : {test_processed.shape}')
train_processed.head()

Train features: (891, 12)
Test features : (418, 11)


,Survived,Pclass,Sex,Age,Fare,Embarked,Title,FamilySize,IsAlone,AgeBand,FareBand,HasCabin
0,0,3,1,22.0,7.2500,2,2,2,0,2,0,0
1,1,1,0,38.0,71.2833,0,3,2,0,3,3,1
2,1,3,0,26.0,7.9250,2,1,1,1,2,1,0
3,1,1,0,35.0,53.1000,2,3,2,0,2,3,1
4,0,3,1,35.0,8.0500,2,2,1,1,2,1,0


In [7]:
# เช็คว่ายังมี missing อยู่ไหม
print('Train:', train_processed.isnull().sum().sum())
print('Test :', test_processed.isnull().sum().sum())

Train: 0
Test : 0


In [8]:
# Split features and target
FEATURES = [c for c in train_processed.columns if c != 'Survived']

X = train_processed[FEATURES].values
y = train_processed['Survived'].values
X_test_final = test_processed[FEATURES].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test_final)

print(f'X_train : {X_train_scaled.shape}')
print(f'X_val   : {X_val_scaled.shape}')
print(f'Features: {FEATURES}')

X_train : (712, 11)
X_val   : (179, 11)
Features: ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'AgeBand', 'FareBand', 'HasCabin']


In [9]:
# SVM
svm_model = SVC(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,
    random_state=42
)
svm_model.fit(X_train_scaled, y_train)

svm_val_pred = svm_model.predict(X_val_scaled)
svm_acc = accuracy_score(y_val, svm_val_pred)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_cv = cross_val_score(svm_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print(f'   SVM Validation Accuracy : {svm_acc:.4f}')
print(f'   CV Accuracy (5-fold)    : {svm_cv.mean():.4f} ± {svm_cv.std():.4f}')

   SVM Validation Accuracy : 0.8324
   CV Accuracy (5-fold)    : 0.8301 ± 0.0279


In [10]:
# ANN
ann_model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)
ann_model.fit(X_train_scaled, y_train)

ann_val_pred = ann_model.predict(X_val_scaled)
ann_acc = accuracy_score(y_val, ann_val_pred)

ann_cv = cross_val_score(ann_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print(f'   ANN Validation Accuracy : {ann_acc:.4f}')
print(f'   CV Accuracy (5-fold)    : {ann_cv.mean():.4f} ± {ann_cv.std():.4f}')

   ANN Validation Accuracy : 0.8268
   CV Accuracy (5-fold)    : 0.8230 ± 0.0320


In [11]:
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    oob_score=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

rf_val_pred = rf_model.predict(X_val_scaled)
rf_acc = accuracy_score(y_val, rf_val_pred)

rf_cv = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print(f'   Random Forest Validation Accuracy : {rf_acc:.4f}')
print(f'   OOB Score                         : {rf_model.oob_score_:.4f}')
print(f'   CV Accuracy (5-fold)              : {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')

   Random Forest Validation Accuracy : 0.8101
   OOB Score                         : 0.8287
   CV Accuracy (5-fold)              : 0.8386 ± 0.0286


In [12]:
# Voting
ensemble = VotingClassifier(
    estimators=[
        ('svm', svm_model),
        ('ann', ann_model),
        ('rf',  rf_model)
    ],
    voting='soft',
    weights=[1, 1, 1.5]
)
ensemble.fit(X_train_scaled, y_train)

ens_val_pred  = ensemble.predict(X_val_scaled)
ens_val_proba = ensemble.predict_proba(X_val_scaled)[:, 1]
ens_acc  = accuracy_score(y_val, ens_val_pred)
ens_auc  = roc_auc_score(y_val, ens_val_proba)

ens_cv = cross_val_score(ensemble, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print(f'   Ensemble Validation Accuracy : {ens_acc:.4f}')
print(f'   ROC-AUC                      : {ens_auc:.4f}')
print(f'   CV Accuracy (5-fold)         : {ens_cv.mean():.4f} ± {ens_cv.std():.4f}')

   Ensemble Validation Accuracy : 0.8436
   ROC-AUC                      : 0.8632
   CV Accuracy (5-fold)         : 0.8272 ± 0.0158


In [13]:
# เปรียบเทียบ model
results = pd.DataFrame({
    'Model': ['SVM', 'ANN', 'Random Forest', 'Ensemble (Soft Voting)'],
    'Val Accuracy': [svm_acc, ann_acc, rf_acc, ens_acc],
    'CV Mean': [svm_cv.mean(), ann_cv.mean(), rf_cv.mean(), ens_cv.mean()],
    'CV Std':  [svm_cv.std(),  ann_cv.std(),  rf_cv.std(),  ens_cv.std()]
})
results = results.sort_values('Val Accuracy', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

                 Model  Val Accuracy  CV Mean   CV Std
Ensemble (Soft Voting)      0.843575 0.827243 0.015823
                   SVM      0.832402 0.830060 0.027916
                   ANN      0.826816 0.823028 0.031955
         Random Forest      0.810056 0.838570 0.028590


In [14]:
# Detailed classification report
print(classification_report(y_val, ens_val_pred,
                             target_names=['Not Survived', 'Survived']))

              precision    recall  f1-score   support

Not Survived       0.87      0.88      0.87       110
    Survived       0.81      0.78      0.79        69

    accuracy                           0.84       179
   macro avg       0.84      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179



In [15]:
# Predict on test set
test_predictions = ensemble.predict(X_test_scaled)

# Create submission dataframe
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived':    test_predictions
})

# Save to CSV
submission.to_csv('submission.csv', index=False)

print(f'   Total predictions: {len(submission)}')
print(f'   Predicted Survived: {submission["Survived"].sum()} ({submission["Survived"].mean():.2%})')
submission.head(10)

   Total predictions: 418
   Predicted Survived: 162 (38.76%)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [16]:
import pickle
with open("ensemble_model.pkl", "wb") as f:
    pickle.dump(ensemble, f)
with open("ensemble_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)